In [12]:
## Arxiv--Research
## Tools creation
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [13]:
## Used the inbuilt tool of wikipedia
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [14]:
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv.name)

arxiv


In [15]:
tools=[wiki,arxiv]

In [16]:
## Custom tools[RAG Tool]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [17]:
loader=WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectordb = FAISS.from_documents(documents, embeddings)
retriever=vectordb.as_retriever()
retriever

C:\Users\apran\AppData\Local\Temp\ipykernel_17036\1105959698.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1527.09it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen imp

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001BEDE0C3CB0>, search_kwargs={})

In [18]:
from langchain_core.tools import create_retriever_tool
retriever_tool = create_retriever_tool(
    retriever,
    "langsmith-search",
    "Search any information about Langsmith"
)

retriever_tool.name

'langsmith-search'

In [19]:
tools=[wiki,arxiv,retriever_tool]

In [20]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\apran\\Desktop\\Langchain\\langvenv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith-search', description='Search any information about Langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001BEDE3D2A20>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001BEE047BE20>)]

In [34]:
## Run all this tools with Agents and LLM Models

## Tools, LLM-->AgentExecutor
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import openai
load_dotenv()
import os

groq_api_key=os.getenv("GROQ_API_KEY")
openai.api_key=os.getenv("OPENAI_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant")

In [35]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

In [39]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt="You are a helpful assistant. Use the langsmith-search tool to answer questions about LangSmith."
)

In [40]:
## Agent Executer
#from langchain.agents import AgentExecutor
#agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
#agent_executor

In [41]:
agent.invoke({
    "messages": [
        {"role": "user", "content": "Tell me about Langsmith"}
    ]
})

{'messages': [HumanMessage(content='Tell me about Langsmith', additional_kwargs={}, response_metadata={}, id='547ff721-630f-46e1-ae18-e0891ce3de85'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 's5eg0ngax', 'function': {'arguments': '{"query":"Langsmith"}', 'name': 'langsmith-search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 248, 'total_tokens': 265, 'completion_time': 0.019502975, 'completion_tokens_details': None, 'prompt_time': 0.014424836, 'prompt_tokens_details': None, 'queue_time': 0.047495654, 'total_time': 0.033927811}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ccc98-6070-7532-8e72-ba0f57a7ee87-0', tool_calls=[{'name': 'langsmith-search', 'args': {'query': 'Langsmith'}, 'id': 's5eg0ngax', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada

In [42]:
agent.invoke({
    "messages": [
        {"role": "user", "content": "What is machine learning?"}
    ]
})

{'messages': [HumanMessage(content='What is machine learning?', additional_kwargs={}, response_metadata={}, id='e1c090fa-1af7-4dca-9332-bb4246f9ad89'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6v21fyy9w', 'function': {'arguments': '{"query":"machine learning definition"}', 'name': 'langsmith-search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 248, 'total_tokens': 266, 'completion_time': 0.023740593, 'completion_tokens_details': None, 'prompt_time': 0.01476117, 'prompt_tokens_details': None, 'queue_time': 0.04574648, 'total_time': 0.038501763}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ccc9b-abac-7fc0-9b2a-91e4e09f62f2-0', tool_calls=[{'name': 'langsmith-search', 'args': {'query': 'machine learning definition'}, 'id': '6v21fyy9w', 'type': 'tool_call'}],

In [43]:
agent.invoke({
    "messages": [
        {"role": "user", "content": "What's the paper 1706.03762 about?"}
    ]
})

{'messages': [HumanMessage(content="What's the paper 1706.03762 about?", additional_kwargs={}, response_metadata={}, id='d2890a1b-ca7e-4ed2-aa54-54f7af495dc1'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'fy8x067kh', 'function': {'arguments': '{"query":"paper 1706.03762 LangSmith"}', 'name': 'langsmith-search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 255, 'total_tokens': 279, 'completion_time': 0.029273815, 'completion_tokens_details': None, 'prompt_time': 0.017704522, 'prompt_tokens_details': None, 'queue_time': 0.047437578, 'total_time': 0.046978337}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ccc9c-2df8-7113-93ff-31fa584a65f4-0', tool_calls=[{'name': 'langsmith-search', 'args': {'query': 'paper 1706.03762 LangSmith'}, 'id': 'fy8x067kh', 'type': 'tool